# Phase 1  — Dataset Preparation and Data Partitioning

This notebook prepares the datasets used throughout the project. The Cleveland dataset is partitioned into **training (70%)**, **validation (15%)**, and **held-out test (15%)** subsets, while the Hungarian and Swiss datasets are preserved as independent external evaluation sets.

The notebook performs preprocessing and in-memory partitioning only. Persistent datasets and metadata are generated later by the project pipeline.

## Data Splitting Strategy

The processed Cleveland dataset does not contain a unique patient identifier. Consequently, true subject-level partitioning cannot be performed. Under the assumption that each record corresponds to a different patient, **StratifiedShuffleSplit** is used to preserve the class distribution across the training, validation, and held-out test sets.

If a future release includes patient identifiers, this procedure should be replaced with subject-aware splitting (e.g., `GroupShuffleSplit`) to guarantee complete patient independence.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit


# Reproducibility configuration
RANDOM_STATE = 42


# Project directory structure
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"


# Original UCI Heart Disease feature names
COLUMN_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope", "ca", "thal", "target",
]

# Predictor columns used for modelling
RAW_FEATURES = [c for c in COLUMN_NAMES if c != "target"]

SITE_FILES = {
    "cleveland": {
        "filename": "processed.cleveland.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data",
    },
    "hungarian": {
        "filename": "processed.hungarian.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.hungarian.data",
    },
    "swiss": {
        "filename": "processed.switzerland.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.switzerland.data",
    },
}

def ensure_raw_files():
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    paths = {}
    for site, info in SITE_FILES.items():
        path = RAW_DATA_DIR / info["filename"]
        paths[site] = path
        if not path.exists() or path.stat().st_size == 0:
            print(f"Downloading {site}...")
            urlretrieve(info["url"], path)
    return paths

def load_site(path, site):
    df = pd.read_csv(path, header=None, names=COLUMN_NAMES, na_values="?")
    for col in COLUMN_NAMES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["target_original"] = df["target"]
    df["target"] = (df["target"] > 0).astype(int)
    df["site"] = site
    df["original_index"] = df.index
    return df

def load_all_sites():
    paths = ensure_raw_files()
    return {site: load_site(path, site) for site, path in paths.items()}

def split_cleveland_v2(cleveland, random_state=RANDOM_STATE):
    # UCI processed Cleveland has no explicit patient ID. Therefore, rows are treated as independent patients.
    # If a patient_id column becomes available later, replace this with GroupShuffleSplit.
    cleveland = cleveland.copy()

    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=random_state)
    rest_idx, test_idx = next(sss_test.split(cleveland, cleveland["target"]))

    rest = cleveland.iloc[rest_idx].copy()
    test = cleveland.iloc[test_idx].copy()

    # Validation is 15% of total. After removing 15% test, validation is 15/85 of the remaining rows.
    val_fraction_of_rest = 0.15 / 0.85
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=val_fraction_of_rest, random_state=random_state)
    train_rel_idx, val_rel_idx = next(sss_val.split(rest, rest["target"]))

    train = rest.iloc[train_rel_idx].copy()
    validation = rest.iloc[val_rel_idx].copy()

    train["partition"] = "train"
    validation["partition"] = "validation"
    test["partition"] = "test"

    return train, validation, test

def verify_no_overlap(train, validation, test):
    sets = {
        "train": set(train["original_index"]),
        "validation": set(validation["original_index"]),
        "test": set(test["original_index"]),
    }
    assert sets["train"].isdisjoint(sets["validation"])
    assert sets["train"].isdisjoint(sets["test"])
    assert sets["validation"].isdisjoint(sets["test"])
    print("No Cleveland original_index overlap across train/validation/test.")

def load_phase1_v2_in_memory():
    sites = load_all_sites()
    train, validation, test = split_cleveland_v2(sites["cleveland"])
    verify_no_overlap(train, validation, test)
    return {
        "cleveland_full": sites["cleveland"],
        "cleveland_train": train,
        "cleveland_validation": validation,
        "cleveland_test": test,
        "hungarian": sites["hungarian"],
        "swiss": sites["swiss"],
    }

## Load the Dataset and Generate the Data Splits

In [2]:
phase1 = load_phase1_v2_in_memory()

for name, df in phase1.items():
    print(f"{name:22s} shape={df.shape}, positive_rate={df['target'].mean():.2%}")

No Cleveland original_index overlap across train/validation/test.
cleveland_full         shape=(303, 17), positive_rate=45.87%
cleveland_train        shape=(211, 18), positive_rate=45.97%
cleveland_validation   shape=(46, 18), positive_rate=45.65%
cleveland_test         shape=(46, 18), positive_rate=45.65%
hungarian              shape=(294, 17), positive_rate=36.05%
swiss                  shape=(123, 17), positive_rate=93.50%


## Validate Split Sizes and Class Distribution

This section verifies that the generated partitions closely match the intended **70/15/15** split while maintaining the original class balance. Minor differences of one sample are expected because the dataset size is not perfectly divisible.

In [3]:
summary_rows = []
for name in ["cleveland_train", "cleveland_validation", "cleveland_test", "hungarian", "swiss"]:
    df = phase1[name]
    summary_rows.append({
        "split": name,
        "rows": len(df),
        "negative": int((df["target"] == 0).sum()),
        "positive": int((df["target"] == 1).sum()),
        "positive_rate": df["target"].mean(),
    })

pd.DataFrame(summary_rows)

,split,rows,negative,positive,positive_rate
0,cleveland_train,211,114,97,0.459716
1,cleveland_validation,46,25,21,0.456522
2,cleveland_test,46,25,21,0.456522
3,hungarian,294,188,106,0.360544
4,swiss,123,8,115,0.934959


## Verify Partition Independence

To ensure data integrity, the original Cleveland row indices are compared across the training, validation, and held-out test partitions. Each original observation should belong to exactly one partition, confirming the absence of data leakage.

In [4]:
verify_no_overlap(
    phase1["cleveland_train"],
    phase1["cleveland_validation"],
    phase1["cleveland_test"],
)

No Cleveland original_index overlap across train/validation/test.
